# CheXpert Plus - Local Image Cache Builder

Downloads the CheXpert Plus chest X-ray dataset from Redivis and packages it into a portable
local cache that every later stage of this project reads from. Running this once removes any
further dependency on the Redivis API and keeps the later notebooks reproducible offline.

## Inputs
- A Redivis API token with access to the `chexpert_plus` dataset
- Internet access enabled on the runtime

## Outputs (written to `chexpert_cache/`)
- `images_partNN.zip` - frontal chest X-rays resized to 336x336, split into ~2 GB shards
- `manifest.csv` - maps every image key to the shard that contains it
- `chexpert_plus_full.parquet` - the complete report and metadata table

## Role in the pipeline
This cache is the single image source for the encoder comparison, the V-RAG dataset and
database build, model finetuning, and evaluation. All later notebooks read images from the
shards by key rather than downloading anything.

## Configuration

All download settings are defined here: how many images to fetch, the output resolution,
download concurrency, shard size, and the Redivis dataset identifiers. The resolution is fixed
at 336 pixels because that is the input size of the LLaVA-1.5 vision encoder, so no later stage
needs to resize again. Worker count is capped to stay under the Redivis request rate limit.

In [1]:
# ============================================================
#  CONFIG — edit here only
# ============================================================
REDIVIS_TOKEN = "AAAGwGJS33f4OxyGTuwmWWyxRfuivWks"   # or set a Kaggle Secret named REDIVIS_API_TOKEN

# What to fetch.
#   None  -> ALL frontal train images (~190k, ~11.6 GB of shards)  [recommended]
#   100000 -> cap it (smaller/faster, but a superset of only *some* future samples)
MAX_IMAGES   = None

IMG_SIZE     = 336        # LLaVA-1.5 CLIP ViT-L/14-336 crop. Do not change.
WORKERS      = 24         # Redivis caps ~1000 req/60s (~16.7 files/s). More threads
                          # past the cap just earn 429s and backoff — 24 is a sane spot.
SHARD_GB     = 2.0        # zip shard size; ~6 shards for the full set. Bump to 20 for one big zip.
OUT_DIR      = "/kaggle/working/chexpert_cache"

# Redivis references (qualified name:id so an upstream rename can't break this)
DATASET_REF  = "chexpert_plus:5yyj"
PNG_TABLE_REF = "PNG_train:s6cj"
REPORT_TABLE  = "df_chexpert_plus_240401:bavj"

# ---- resume ----
# Kaggle wipes /kaggle/working between sessions. If the 12h clock beats you:
#   1. Save/commit this notebook (shards land in its Output)
#   2. New session -> Add Data -> this notebook's output
#   3. Set RESUME_FROM below to that path. Finished shards are skipped.
RESUME_FROM  = None       # e.g. "/kaggle/input/chexpert-cache-part1/chexpert_cache"

import os
from pathlib import Path
OUT = Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)

try:                                            # prefer a Kaggle Secret if you set one
    from kaggle_secrets import UserSecretsClient
    REDIVIS_TOKEN = UserSecretsClient().get_secret("REDIVIS_API_TOKEN") or REDIVIS_TOKEN
    print("using REDIVIS_API_TOKEN from Kaggle Secrets")
except Exception:
    print("using the token hardcoded in this cell")
os.environ["REDIVIS_API_TOKEN"] = REDIVIS_TOKEN

import shutil
free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"/kaggle/working free: {free_gb:.1f} GB   (need ~12 GB for the full set)")
if free_gb < 14:
    print("tight on disk — consider lowering MAX_IMAGES")
print("Settings -> Internet must be ON, or every request below fails.")


using the token hardcoded in this cell
/kaggle/working free: 20.9 GB   (need ~12 GB for the full set)
Settings -> Internet must be ON, or every request below fails.


## Install

Installs the Redivis client, the API used to access CheXpert Plus.

In [2]:
!pip install redivis -q
print('redivis installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.2/75.2 kB 950.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.9/213.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.7/786.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.0/396.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.1 MB/s eta 0:00:00
redivis installed


## Connect to Redivis

Opens the API connection and defines a download helper that retries on truncated responses.
Redivis rate-limits requests, so the helper backs off and resumes instead of failing the run.

In [3]:
# ============================================================
#  Connect to Redivis  (+ the short-read fix this whole thing depends on)
# ============================================================
import io, redivis
import redivis.common.TabularReader as _TR

# ---------------------------------------------------------------------------
#  Without this you get, from inside redivis (not your code):
#     OSError: Expected to be able to read 566984 bytes for message body, got 11853
#
#  to_directory() hands the raw urllib3 socket stream straight to pyarrow. The API
#  serves the file listing with `x-accel-buffering: no`, so the body arrives in
#  small unbuffered chunks; urllib3's read(n) returns only what has landed in the
#  socket so far (~12 KB), pyarrow wants a ~567 KB Arrow message, sees the short
#  read, and calls the stream truncated. It's a race with the network — it can
#  "work" one day and fail the next.
#
#  Fix: for the /rawFiles listing only, drain to EOF and give pyarrow a complete
#  seekable buffer.
# ---------------------------------------------------------------------------
if not getattr(_TR, "_shortread_patched", False):
    _orig = _TR.make_request
    def _buffered(*a, **k):
        r = _orig(*a, **k)
        p = k.get("path", "") or ""
        if isinstance(p, str) and p.endswith("/rawFiles") and getattr(r, "status_code", None) == 200:
            r.raw = io.BytesIO(r.raw.read(decode_content=True))
        return r
    _TR.make_request = _buffered
    _TR._shortread_patched = True
    print("redivis short-read patch applied")

dataset = redivis.organization("AIMI").dataset(DATASET_REF)
dataset.get()
print(f"{dataset.properties.get('name')}  version {dataset.properties.get('version', {}).get('tag')}")


def to_png_key(path_to_image: str) -> str:
    """
    Report table stores : train/patient00003/study1/view1_frontal.jpg
    PNG_train stores    : patient00003/study1/view1_frontal.png
    (no split prefix, .png not .jpg — this mapping is the whole ballgame)
    """
    parts = list(Path(str(path_to_image).strip()).parts)
    while parts and (parts[0] in ("train", "valid", "test") or parts[0].startswith("CheXpert")):
        parts = parts[1:]
    return str(Path(*parts).with_suffix(".png"))


from PIL import Image
def clip_preprocess(img, size=IMG_SIZE):
    """
    Byte-for-byte what llava-1.5-7b-hf's CLIPImageProcessor does
    (preprocessor_config.json: shortest_edge 336 -> bicubic -> center crop 336).
    Verified against the real processor: max|diff| == 0.000000 on every CheXpert size.

    NOTE the int() below, not round(). transformers computes the long edge as
    int(size * long / short) — truncation. Rounding instead gives a 410px-wide
    intermediate for a 2828x2320 image where transformers gives 409, which shifts
    the center crop by a pixel. 2828x2320 is the most common CheXpert size, so that
    one-character difference would have skewed most of the cache.

    Kept grayscale: X-rays are single-channel, and resizing in 'L' then replicating
    to RGB is bit-identical to resizing in RGB (verified) at ~61 KB/image vs ~103 KB.
    """
    img = img.convert("L")
    w, h = img.size
    short, lng = (w, h) if w <= h else (h, w)
    new_short, new_long = size, int(size * lng / short)
    new_w, new_h = (new_short, new_long) if w <= h else (new_long, new_short)
    img = img.resize((new_w, new_h), Image.BICUBIC)
    left, top = (new_w - size) // 2, (new_h - size) // 2
    return img.crop((left, top, left + size, top + size))

print(f"   key map: train/patient00003/study1/view1_frontal.jpg -> {to_png_key('train/patient00003/study1/view1_frontal.jpg')}")


redivis short-read patch applied
Please delete the token on Redivis and remove it from your code, and follow the authentication prompts here instead.

This environment variable should only ever be set in a non-interactive environment, such as in an automated script or service.

CheXpert Plus  version v1.0
   key map: train/patient00003/study1/view1_frontal.jpg -> patient00003/study1/view1_frontal.png


## Download the tabular data

Retrieves the full record table (report text, patient identifier, view position, dataset split)
and stores it as a single parquet file. Later notebooks read patient and report information
from this file rather than querying the API again.

In [4]:
# ============================================================
#  1) The tabular data — FULL table, every row, every column
# ============================================================
# Saved first and on its own: it's small and fast, so even if the image download
# later dies you already have the reports. This is the file that replaces the
# Redivis query in the training notebook.
import pandas as pd, time

FULL_TABLE_QUERY = f"""
SELECT
    path_to_image, path_to_dcm, deid_patient_id, patient_report_date_order,
    frontal_lateral, ap_pa, split, report,
    section_narrative, section_clinical_history, section_history,
    section_comparison, section_technique, section_procedure_comments,
    section_findings, section_impression, section_end_of_impression,
    section_summary, section_accession_number
FROM `{REPORT_TABLE}`
"""

print("pulling the full report table ...")
t0 = time.time()
raw_df = dataset.query(FULL_TABLE_QUERY).to_pandas_dataframe()
print(f"{raw_df.shape[0]:,} rows x {raw_df.shape[1]} cols in {time.time()-t0:.0f}s")

parquet_path = OUT / "chexpert_plus_full.parquet"
raw_df.to_parquet(parquet_path, index=False, compression="snappy")
print(f"saved {parquet_path}  ({parquet_path.stat().st_size/1e6:.0f} MB)")
print("\nsplits:", raw_df['split'].str.lower().str.strip().value_counts().to_dict())
print("views :", raw_df['frontal_lateral'].str.lower().str.strip().value_counts().to_dict())


pulling the full report table ...
Please delete the token on Redivis and remove it from your code, and follow the authentication prompts here instead.

This environment variable should only ever be set in a non-interactive environment, such as in an automated script or service.



  0%|          | 0/223462 [00:00<?, ?it/s]

223,462 rows x 19 cols in 20s
saved /kaggle/working/chexpert_cache/chexpert_plus_full.parquet  (114 MB)

splits: {'train': 223228, 'valid': 234}
views : {'frontal': 191071, 'lateral': 32391}


## Select the images to download

Filters the table down to the images actually needed: frontal views from the training split.
Lateral views and other splits are excluded because every model in this project operates on
frontal chest X-rays.

In [5]:
# ============================================================
#  2) Which images to fetch
# ============================================================
# ALL frontal train images — deliberately NOT the 100k training sample.
#
# The training notebook drops test_vqa patients BEFORE df.sample(), so the sampled
# set depends on the blacklist, the filters and the seed. Cache only that 100k and
# any future change (different seed, different SAMPLE_SIZE, one of the three V-RAG
# finetunes using a different slice) sends you straight back to Redivis.
#
# A superset of every frontal train image is immune to all of that. It costs 11.6 GB
# instead of 6.1 GB. That is the price of "never touch Redivis again".

df = raw_df.copy()
df = df[df['frontal_lateral'].str.lower().str.strip() == 'frontal']
df = df[df['split'].str.lower().str.strip() == 'train']
print(f"frontal + train rows: {len(df):,}")

keys = sorted({to_png_key(p) for p in df['path_to_image']})
print(f"unique image keys   : {len(keys):,}")

if MAX_IMAGES:
    keys = keys[:MAX_IMAGES]
    print(f"capped to           : {len(keys):,}  (MAX_IMAGES)")

print(f"\nover the wire : ~{len(keys)*3.29/1000:.0f} GB")
print(f"on disk after resize: ~{len(keys)*61/1e6:.1f} GB")
print(f"floor at Redivis' ~16.7 files/s cap: {len(keys)/16.7/3600:.1f} h")
print(f"at a realistic 8 files/s          : {len(keys)/8/3600:.1f} h  (fits a 12h session)")


frontal + train rows: 190,869
unique image keys   : 190,869

over the wire : ~628 GB
on disk after resize: ~11.6 GB
floor at Redivis' ~16.7 files/s cap: 3.2 h
at a realistic 8 files/s          : 6.6 h  (fits a 12h session)


## Download, resize and shard

Downloads each image, resizes it to the target resolution, and writes it into fixed-size zip
shards. Sharding keeps individual files small enough to upload and mount on other platforms,
and the manifest records which shard holds each image so a single image can be read later
without unpacking the whole archive. This is the long-running step and supports resuming.

In [6]:
# ============================================================
#  3) Download -> resize -> ZIP shards
# ============================================================
#  Written straight into zips, with no intermediate loose files. That's not a
#  style choice: /kaggle/working is ~20 GB, and 11.6 GB of images + an 11.6 GB
#  zip *of* those images is 23 GB. It would not fit. Streaming into the zip keeps
#  peak disk at ~11.6 GB.
#
#  Sharded at SHARD_GB rather than one giant file: each shard is independently
#  valid, so a corrupt or half-downloaded file costs you 2 GB, not 12. Kaggle's
#  "Download All" wraps them into a single archive anyway.
import zipfile, csv, traceback
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

png_table = dataset.table(PNG_TABLE_REF); png_table.get()

# Resolve every File handle ONCE, single-threaded. File.read() hits a per-file-id
# endpoint and parallelises safely; building the directory inside threads would not.
print("indexing PNG_train (one time, ~30-60s) ...")
t0 = time.time()
by_key = {str(f.path): f for f in png_table.list_files()}
print(f"{len(by_key):,} handles in {time.time()-t0:.0f}s")

# ---- resume: skip anything already inside a shard ----
done = set()
for d in [OUT] + ([Path(RESUME_FROM)] if RESUME_FROM else []):
    if not Path(d).exists(): continue
    for z in sorted(Path(d).glob("*.zip")):
        try:
            with zipfile.ZipFile(z) as zf:
                done |= {n for n in zf.namelist() if n.endswith(".png")}
            print(f"   resume: {z.name} -> {len(done):,} already fetched")
        except Exception as e:
            print(f"   unreadable/partial shard {z.name}: {e} (delete it and re-run)")

todo    = [k for k in keys if k not in done and k in by_key]
missing = [k for k in keys if k not in by_key]
print(f"\nto fetch: {len(todo):,} | already have: {len(done):,} | not in PNG_train: {len(missing):,}")

SHARD_BYTES = int(SHARD_GB * 1e9)
existing    = sorted(OUT.glob("images_part*.zip"))
part        = (int(existing[-1].stem.replace("images_part","")) + 1) if existing else 0

def fetch(key):
    for attempt in range(4):
        try:
            raw = by_key[key].read()
            buf = io.BytesIO()
            clip_preprocess(Image.open(io.BytesIO(raw))).save(buf, format="PNG")
            return key, buf.getvalue(), len(raw)
        except Exception as e:
            if type(e).__name__ == "NotFoundError":
                return key, None, 0
            if attempt == 3:
                return key, f"ERR:{type(e).__name__}: {e}", 0
            time.sleep(1.5 * (attempt + 1))
    return key, None, 0

manifest, failures = [], []
wire = shard_sz = ok = 0
# ZIP_STORED: PNG is already deflate-compressed, so zip compression buys ~0% for real CPU.
zf = zipfile.ZipFile(OUT / f"images_part{part:02d}.zip", "a", zipfile.ZIP_STORED, allowZip64=True)
t0 = time.time()

print(f"\nDownloading with {WORKERS} threads — live progress below.", flush=True)
print("   (first images land within seconds; the bar updates continuously)\n", flush=True)

try:
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = [ex.submit(fetch, k) for k in todo]
        bar  = tqdm(as_completed(futs), total=len(todo), desc="fetch", unit="img", smoothing=0.05)
        for n, fut in enumerate(bar, 1):
            key, data, nraw = fut.result()
            if not isinstance(data, (bytes, bytearray)):
                failures.append((key, data or "not found")); continue

            # single writer (this thread) -> no lock needed, downloads stay parallel
            zf.writestr(key, data)
            manifest.append((key, f"images_part{part:02d}.zip"))
            shard_sz += len(data); wire += nraw; ok += 1

            if shard_sz >= SHARD_BYTES:                      # rotate
                zf.close(); print(f"   images_part{part:02d}.zip sealed ({shard_sz/1e9:.2f} GB)", flush=True)
                part += 1; shard_sz = 0
                zf = zipfile.ZipFile(OUT / f"images_part{part:02d}.zip", "a", zipfile.ZIP_STORED, allowZip64=True)

            if n % 200 == 0:
                el = time.time() - t0
                bar.set_postfix_str(f"{wire/el/1e6:.0f} MB/s wire | {ok:,} ok | {len(failures)} fail "
                                    f"| part{part:02d}", refresh=False)
finally:
    zf.close()
    with open(OUT / "manifest.csv", "a", newline="") as f:
        csv.writer(f).writerows(manifest)

el = time.time() - t0
print(f"\nfetched {ok:,} in {el/3600:.2f}h  ({ok/max(el,1):.1f} img/s, {wire/max(el,1)/1e6:.1f} MB/s wire)")
print(f"   failed: {len(failures):,}")
for k, e in failures[:5]:
    print(f"     {k}: {e}")
if failures:
    print("   -> just re-run this cell; it resumes and retries only what's missing")


indexing PNG_train (one time, ~30-60s) ...
223,228 handles in 32s

to fetch: 190,869 | already have: 0 | not in PNG_train: 0

   (first images land within seconds; the bar updates continuously)



fetch:   0%|          | 0/190869 [00:00<?, ?img/s]

   images_part00.zip sealed (2.00 GB)
   images_part01.zip sealed (2.00 GB)
   images_part02.zip sealed (2.00 GB)
   images_part03.zip sealed (2.00 GB)
   images_part04.zip sealed (2.00 GB)

fetched 190,868 in 9.23h  (5.7 img/s, 19.9 MB/s wire)
   failed: 1
     patient32368/study1/view1_frontal.png: ERR:OSError: unrecognized data stream contents when reading image file
   -> just re-run this cell; it resumes and retries only what's missing


## Verify the cache

Confirms the shard count, the number of images written, and that manifest entries resolve to
real files. A truncated or interrupted download is caught here rather than surfacing as a
missing-image error in a later notebook.

In [7]:

import numpy as np, random as _rnd

zips  = sorted(OUT.glob("images_part*.zip"))
names = set()
for z in zips:
    with zipfile.ZipFile(z) as zf:
        names |= {n for n in zf.namelist() if n.endswith(".png")}
total = sum(z.stat().st_size for z in zips)

print(f"shards        : {len(zips)}")
print(f"images inside : {len(names):,}")
print(f"images total  : {total/1e9:.2f} GB")
print(f"parquet       : {(OUT/'chexpert_plus_full.parquet').stat().st_size/1e6:.0f} MB")

# spot-check: real pixels, right size, right layout?
print("\nspot-checking 5 random images:")
bad = 0
with zipfile.ZipFile(_rnd.choice(zips)) as zf:
    members = [n for n in zf.namelist() if n.endswith(".png")]
    for name in _rnd.sample(members, min(5, len(members))):
        img = Image.open(io.BytesIO(zf.read(name)))
        a   = np.asarray(img)
        okay = img.size == (IMG_SIZE, IMG_SIZE) and a.std() > 1.0
        bad += (not okay)
        print(f"   {name:46s} {img.size} {img.mode} std={a.std():5.1f} {'' if okay else 'BLANK/WRONG SIZE'}")

gaps = [k for k in keys if k not in names]
print(f"\nrequested but missing from shards: {len(gaps):,}")
print("\n" + "="*60)
if bad or gaps:
    print("re-run the download cell — it resumes and fills the gaps")
else:
    print("COMPLETE. Download the Output folder (Kaggle zips it for you).")
    print("   You never need to touch Redivis again.")
print("="*60)
for f in sorted(OUT.iterdir()):
    print(f"   {f.name:34s} {f.stat().st_size/1e6:9.1f} MB")


shards        : 6
images inside : 190,868
images total  : 11.40 GB
parquet       : 114 MB

spot-checking 5 random images:
   patient34797/study7/view1_frontal.png          (336, 336) L std= 66.7 
   patient34050/study1/view1_frontal.png          (336, 336) L std= 70.1 
   patient33374/study7/view1_frontal.png          (336, 336) L std= 70.2 
   patient35025/study3/view1_frontal.png          (336, 336) L std= 70.8 
   patient33372/study1/view1_frontal.png          (336, 336) L std= 73.6 

requested but missing from shards: 1

re-run the download cell — it resumes and fills the gaps
   chexpert_plus_full.parquet             114.4 MB
   images_part00.zip                     2005.1 MB
   images_part01.zip                     2005.1 MB
   images_part02.zip                     2005.1 MB
   images_part03.zip                     2005.0 MB
   images_part04.zip                     2005.1 MB
   images_part05.zip                     1377.8 MB
   manifest.csv                            10.9 MB


## Using the cache downstream

Mount or copy `chexpert_cache/` wherever the later notebooks run. They locate it by looking for
the `images_part*` shards and read individual images by key through `manifest.csv`, so the
shards stay compressed and no extraction step is required.